# 📊 交互式数据分析面板

本 Notebook 基于 5000 条模拟电商订单数据，展示多维度的交互式分析。

**技术栈**: DuckDB（数据引擎）+ Plotly（交互图表）+ ipywidgets（筛选控件）

**使用方式**: 点击顶部菜单 `Run All` 执行所有 cell，然后通过下拉框和滑块探索数据。

---

## 0. 环境初始化

加载依赖、配置全局图表主题和控件样式，创建模拟订单数据。

In [ ]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

COLORS = ['#6366f1', '#22c55e', '#f59e0b', '#ef4444', '#8b5cf6', '#06b6d4']

custom_template = go.layout.Template(
    layout=go.Layout(
        font=dict(family='Inter, -apple-system, sans-serif', size=13, color='#1f2937'),
        title=dict(font=dict(size=15, color='#111827'), x=0.02, xanchor='left'),
        plot_bgcolor='#fafbfc',
        paper_bgcolor='white',
        colorway=COLORS,
        xaxis=dict(gridcolor='#e5e7eb', linecolor='#d1d5db', linewidth=1, zeroline=False),
        yaxis=dict(gridcolor='#e5e7eb', linecolor='#d1d5db', linewidth=1, zeroline=False),
        legend=dict(bgcolor='rgba(255,255,255,0.8)', bordercolor='#e5e7eb', borderwidth=1, font=dict(size=11)),
        hoverlabel=dict(bgcolor='white', bordercolor='#d1d5db', font=dict(family='Inter, sans-serif', size=12, color='#1f2937')),
        margin=dict(l=60, r=30, t=60, b=50),
    )
)
pio.templates['custom'] = custom_template
pio.templates.default = 'custom'

DD_LAYOUT = widgets.Layout(width='180px')
DD_STYLE = {'description_width': '60px'}
SLIDER_LAYOUT = widgets.Layout(width='260px')
SLIDER_STYLE = {'description_width': '70px'}
BOX_LAYOUT = widgets.Layout(padding='10px 16px', gap='12px', border='1px solid #e5e7eb', border_radius='8px', margin='0 0 8px 0', background='#f9fafb')

print('✅ 主题和样式配置完成')

In [ ]:
conn = duckdb.connect()
conn.execute('''
CREATE OR REPLACE TABLE orders AS
SELECT
    row_number() OVER () AS order_id,
    '2025-01-01'::DATE + INTERVAL (floor(random() * 90)::INT) DAY AS order_date,
    CASE floor(random() * 4)::INT
        WHEN 0 THEN '直接访问' WHEN 1 THEN '搜索引擎'
        WHEN 2 THEN '社交媒体' ELSE '付费广告'
    END AS channel,
    CASE floor(random() * 3)::INT
        WHEN 0 THEN '电子产品' WHEN 1 THEN '服装' ELSE '食品'
    END AS category,
    round(random() * 500 + 10, 2) AS amount,
    CASE WHEN random() > 0.15 THEN '已完成' ELSE '已退款' END AS status
FROM generate_series(1, 5000)
''')

ALL_ORDERS = conn.execute('SELECT * FROM orders').fetchdf()
CHANNELS = ['全部'] + sorted(ALL_ORDERS['channel'].unique().tolist())
CATEGORIES = ['全部'] + sorted(ALL_ORDERS['category'].unique().tolist())

print(f'✅ 数据就绪: {len(ALL_ORDERS)} 条订单')
ALL_ORDERS.head()

---

## 1. 每日收入趋势

柱状图显示每日收入，红色虚线为 7 日移动均线。鼠标悬停查看详情，拖拽缩放聚焦某个时间段。

In [ ]:
completed = ALL_ORDERS[ALL_ORDERS['status'] == '已完成']
daily = completed.groupby('order_date').agg(
    order_count=('amount', 'count'), total_revenue=('amount', 'sum')
).reset_index().sort_values('order_date')
daily['ma7'] = daily['total_revenue'].rolling(7).mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('每日收入趋势', '每日订单量'))
fig.add_trace(go.Bar(x=daily['order_date'], y=daily['total_revenue'], name='每日收入',
    marker_color=COLORS[0], opacity=0.65, hovertemplate='%{x|%m-%d}<br>收入: ¥%{y:,.0f}<extra></extra>'), row=1, col=1)
fig.add_trace(go.Scatter(x=daily['order_date'], y=daily['ma7'], name='7日均线',
    line=dict(color=COLORS[3], width=2.5, dash='dot'), hovertemplate='%{x|%m-%d}<br>均线: ¥%{y:,.0f}<extra></extra>'), row=1, col=1)
fig.add_trace(go.Scatter(x=daily['order_date'], y=daily['order_count'], name='订单量',
    fill='tozeroy', fillcolor='rgba(34,197,94,0.15)', line=dict(color=COLORS[1], width=1.8),
    hovertemplate='%{x|%m-%d}<br>订单: %{y}<extra></extra>'), row=2, col=1)

peak = daily.loc[daily['total_revenue'].idxmax()]
fig.add_annotation(x=peak['order_date'], y=peak['total_revenue'],
    text=f'峰值 ¥{peak["total_revenue"]:,.0f}', showarrow=True, arrowhead=2,
    arrowcolor=COLORS[3], font=dict(size=11, color=COLORS[3]), row=1, col=1)

total = daily['total_revenue'].sum()
fig.update_layout(height=580, title_text=f'总收入 ¥{total:,.0f} · 日均 ¥{daily["total_revenue"].mean():,.0f} · 总订单 {daily["order_count"].sum():,}')
fig.update_yaxes(title_text='收入 (¥)', row=1, col=1)
fig.update_yaxes(title_text='订单量', row=2, col=1)
fig.show()

---

## 2. 渠道分析

左侧环形图展示收入占比，右侧柱状图对比平均客单价。

In [ ]:
ch = ALL_ORDERS.groupby('channel').agg(
    order_count=('order_id', 'count'), revenue=('amount', 'sum'), avg_amount=('amount', 'mean')
).reset_index().sort_values('revenue', ascending=False)

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'pie'}, {'type': 'bar'}]],
                    subplot_titles=('各渠道收入占比', '各渠道平均客单价'))
fig.add_trace(go.Pie(labels=ch['channel'], values=ch['revenue'], marker_colors=COLORS[:len(ch)],
    textinfo='label+percent', textfont=dict(size=12), hole=0.35), row=1, col=1)
fig.add_trace(go.Bar(x=ch['channel'], y=ch['avg_amount'], marker_color=COLORS[:len(ch)],
    text=ch['avg_amount'].apply(lambda v: f'¥{v:.0f}'), textposition='outside', textfont=dict(size=11)), row=1, col=2)
fig.update_layout(height=420, showlegend=False)
fig.update_yaxes(title_text='平均客单价 (¥)', row=1, col=2)
fig.show()

---

## 3. 各渠道收入趋势对比

四个渠道拆分为独立子图，方便观察趋势差异。

In [ ]:
daily_ch = ALL_ORDERS[ALL_ORDERS['status'] == '已完成'].groupby(['order_date', 'channel']).agg(
    revenue=('amount', 'sum')).reset_index()
fig = px.line(daily_ch, x='order_date', y='revenue', facet_col='channel', facet_col_wrap=2,
              color='channel', title='各渠道每日收入趋势',
              labels={'order_date': '日期', 'revenue': '收入 (¥)', 'channel': '渠道'},
              color_discrete_sequence=COLORS)
fig.update_layout(height=480, showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1], font=dict(size=12)))
fig.show()

---

## 4. 品类 × 渠道交叉分析

热力图展示品类在各渠道上的收入分布，颜色越深代表收入越高。

In [ ]:
cross = ALL_ORDERS[ALL_ORDERS['status'] == '已完成'].groupby(['category', 'channel']).agg(
    revenue=('amount', 'sum')).reset_index()
pivot = cross.pivot(index='category', columns='channel', values='revenue').fillna(0)
fig = px.imshow(pivot, text_auto=',.0f', aspect='auto', color_continuous_scale='Purples',
                title='品类 × 渠道 收入热力图', labels=dict(x='渠道', y='品类', color='收入 (¥)'))
fig.update_layout(height=340)
fig.update_traces(textfont=dict(size=13))
fig.show()

---

## 5. 🎛️ 收入趋势筛选面板

- **渠道** — 聚焦某个流量来源
- **品类** — 聚焦某个商品类目
- **状态** — 切换已完成/已退款订单
- **均线天数** — 调整移动均线的平滑窗口（3~30天）

In [ ]:
w_ch = widgets.Dropdown(options=CHANNELS, value='全部', description='渠道:', layout=DD_LAYOUT, style=DD_STYLE)
w_cat = widgets.Dropdown(options=CATEGORIES, value='全部', description='品类:', layout=DD_LAYOUT, style=DD_STYLE)
w_st = widgets.Dropdown(options=['全部', '已完成', '已退款'], value='已完成', description='状态:', layout=DD_LAYOUT, style=DD_STYLE)
w_ma = widgets.IntSlider(value=7, min=3, max=30, step=1, description='均线天数:', layout=SLIDER_LAYOUT, style=SLIDER_STYLE)

# 用 FigureWidget 避免重复渲染
fw1 = go.FigureWidget(make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                                     subplot_titles=('每日收入趋势', '每日订单量')))
fw1.update_layout(height=540, template='custom')
fw1.update_yaxes(title_text='收入 (¥)', row=1, col=1)
fw1.update_yaxes(title_text='订单量', row=2, col=1)

# 预添加 traces（后续只更新数据）
fw1.add_trace(go.Bar(name='每日收入', marker_color=COLORS[0], opacity=0.65,
    hovertemplate='%{x|%m-%d}<br>收入: ¥%{y:,.0f}<extra></extra>'), row=1, col=1)
fw1.add_trace(go.Scatter(name='均线', line=dict(color=COLORS[3], width=2.5, dash='dot'),
    hovertemplate='%{x|%m-%d}<br>均线: ¥%{y:,.0f}<extra></extra>'), row=1, col=1)
fw1.add_trace(go.Scatter(name='订单量', fill='tozeroy', fillcolor='rgba(34,197,94,0.15)',
    line=dict(color=COLORS[1], width=1.8),
    hovertemplate='%{x|%m-%d}<br>订单: %{y}<extra></extra>'), row=2, col=1)

def update_revenue(*args):
    df = ALL_ORDERS.copy()
    if w_ch.value != '全部': df = df[df['channel'] == w_ch.value]
    if w_cat.value != '全部': df = df[df['category'] == w_cat.value]
    if w_st.value != '全部': df = df[df['status'] == w_st.value]

    if df.empty:
        with fw1.batch_update():
            for t in fw1.data: t.x, t.y = [], []
            fw1.layout.title.text = '⚠️ 无数据'
        return

    d = df.groupby('order_date').agg(order_count=('amount', 'count'), total_revenue=('amount', 'sum')
    ).reset_index().sort_values('order_date')
    ma = d['total_revenue'].rolling(w_ma.value).mean()

    parts = [f'{k}={v}' for k, v in [('渠道', w_ch.value), ('品类', w_cat.value), ('状态', w_st.value)] if v != '全部']
    suffix = f' · {" · ".join(parts)}' if parts else ''

    with fw1.batch_update():
        fw1.data[0].x, fw1.data[0].y = d['order_date'], d['total_revenue']
        fw1.data[1].x, fw1.data[1].y = d['order_date'], ma
        fw1.data[1].name = f'{w_ma.value}日均线'
        fw1.data[2].x, fw1.data[2].y = d['order_date'], d['order_count']
        total = d['total_revenue'].sum()
        fw1.layout.title.text = f'总收入 ¥{total:,.0f} · 日均 ¥{d["total_revenue"].mean():,.0f} · 总订单 {d["order_count"].sum():,}{suffix}'

for w in [w_ch, w_cat, w_st, w_ma]:
    w.observe(update_revenue, names='value')

update_revenue()  # 初始渲染
controls1 = widgets.HBox([w_ch, w_cat, w_st, w_ma], layout=BOX_LAYOUT)
display(controls1, fw1)

---

## 6. 🎛️ 渠道对比筛选面板

- **总收入** — 各渠道的收入贡献
- **订单量** — 各渠道的订单规模
- **平均客单价** — 各渠道的用户消费水平
- **退款率** — 各渠道的退款风险

In [ ]:
METRIC_MAP = {'总收入': 'revenue', '订单量': 'order_count', '平均客单价': 'avg_amount', '退款率 %': 'refund_pct'}

w_metric = widgets.Dropdown(options=list(METRIC_MAP.keys()), value='总收入', description='指标:', layout=DD_LAYOUT, style=DD_STYLE)
w_cat2 = widgets.Dropdown(options=CATEGORIES, value='全部', description='品类:', layout=DD_LAYOUT, style=DD_STYLE)

# 预创建 FigureWidget
fw2 = go.FigureWidget(make_subplots(rows=1, cols=2, specs=[[{'type': 'pie'}, {'type': 'bar'}]],
                                       subplot_titles=('占比', '对比')))
fw2.update_layout(height=400, showlegend=False, template='custom')
fw2.add_trace(go.Pie(hole=0.35, textinfo='label+percent', textfont=dict(size=12)), row=1, col=1)
fw2.add_trace(go.Bar(textposition='outside', textfont=dict(size=11)), row=1, col=2)

def update_channel(*args):
    df = ALL_ORDERS.copy()
    if w_cat2.value != '全部': df = df[df['category'] == w_cat2.value]
    if df.empty:
        with fw2.batch_update():
            fw2.data[0].labels, fw2.data[0].values = [], []
            fw2.data[1].x, fw2.data[1].y = [], []
        return

    agg = df.groupby('channel').agg(order_count=('order_id', 'count'), revenue=('amount', 'sum'),
        avg_amount=('amount', 'mean')).reset_index()
    refund = df[df['status'] == '已退款'].groupby('channel')['amount'].sum().reset_index()
    refund.columns = ['channel', 'refund_amount']
    agg = agg.merge(refund, on='channel', how='left').fillna(0)
    agg['refund_pct'] = (agg['refund_amount'] / agg['revenue'] * 100).round(1)
    agg = agg.sort_values('revenue', ascending=False)

    col = METRIC_MAP[w_metric.value]
    cat_suffix = f' · {w_cat2.value}' if w_cat2.value != '全部' else ''
    fmt_fn = (lambda v: f'{v:.1f}%') if col == 'refund_pct' else (lambda v: f'{v:,.0f}')

    with fw2.batch_update():
        fw2.data[0].labels = agg['channel']
        fw2.data[0].values = agg[col]
        fw2.data[0].marker.colors = COLORS[:len(agg)]
        fw2.data[1].x = agg['channel']
        fw2.data[1].y = agg[col]
        fw2.data[1].marker.color = COLORS[:len(agg)]
        fw2.data[1].text = [fmt_fn(v) for v in agg[col]]
        fw2.layout.annotations[0].text = f'各渠道{w_metric.value}占比{cat_suffix}'
        fw2.layout.annotations[1].text = f'各渠道{w_metric.value}对比{cat_suffix}'

for w in [w_metric, w_cat2]:
    w.observe(update_channel, names='value')

update_channel()  # 初始渲染
controls2 = widgets.HBox([w_metric, w_cat2], layout=BOX_LAYOUT)
display(controls2, fw2)